In [1]:
import os
import torch
from transformers import BertTokenizerFast, BertModel
from data import *
from train import *
from model import *
from evaluate import *

In [2]:
# Khởi tạo tokenizer và model BERT
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer))  # Resize embedding BERT

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30526, 768, padding_idx=0)

# Load data

In [3]:
data_info = loadMATRES()

Label Mapping: {'AFTER': 0, 'BEFORE': 1, 'EQUAL': 2, 'VAGUE': 3}


In [4]:
train_df = data_info['train_df']
valid_df = data_info['valid_df']
test_df = data_info['test_df']
num_classes = data_info['num_classes']
relation_map = data_info['label_mapping']
id2label = data_info['id2label']

type_map = {"UNKNOWN": 0}
num_types = 1
rule_filename = 'rules_MATRES.txt'

cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/MATRES/cache/'

os.makedirs(cache_path + "train/", exist_ok=True)
os.makedirs(cache_path + "test/", exist_ok=True)
os.makedirs(cache_path + "valid/", exist_ok=True)

train_token_cache_path = cache_path + "train/token_cache_bert.pt"
train_spacy_jsonl_path = cache_path + "train/spacy_cache_bert.jsonl"

test_token_cache_path = cache_path + "test/token_cache_bert.pt"
test_spacy_jsonl_path = cache_path + "test/spacy_cache_bert.jsonl"

valid_token_cache_path = cache_path + "valid/token_cache_bert.pt"
valid_spacy_jsonl_path = cache_path + "valid/spacy_cache_bert.jsonl"

batch_size = 128
window_size = 128
MAX_LENGTH = 256

In [5]:
relation_map

{'AFTER': 0, 'BEFORE': 1, 'EQUAL': 2, 'VAGUE': 3}

In [6]:
train_loader = create_dataloader(train_token_cache_path, train_spacy_jsonl_path, batch_size=batch_size)
val_loader   = create_dataloader(valid_token_cache_path, valid_spacy_jsonl_path, batch_size=batch_size)
test_loader  = create_dataloader(test_token_cache_path, test_spacy_jsonl_path, batch_size=batch_size)

# Create Model

In [7]:
mode           = 'baseline' # 'baseline_reasoning' # 'baseline'
enc_layers     = 2
nhead          = 8
p_drop         = 0.1
step_size      = 10.0
lambda_kl      = 0.0001
smooth_tau     = 0.1
lambda_entropy = 0.1
max_steps      = 20
epochs         = 40
lr             = 0.0001
lambda_distill = 1.0
lambda_energy  = 0.5
use_amp        = True
freeze_bert    = True
use_deq        = False
output_option  = 'direct_q' # 'direct_q' or 'refinement'
# save_path      = f"/data/ddao/TRE/pretrained_models/Reasoning/i2b2/{mode}_BERT_{output_option}.pt"
save_path      = f"/data/ddao/TRE/pretrained_models/Reasoning/MATRES/{mode}_BERT.pt"

In [8]:
model = create_model(
    mode=mode,
    bert=bert,
    num_classes=num_classes,
    num_types=num_types,
    type_map=type_map,
    relation_map=relation_map,
    enc_layers=enc_layers,
    nhead=nhead,
    p_drop=p_drop,
    step_size=step_size,
    lambda_kl=lambda_kl,
    smooth_tau=smooth_tau,
    lambda_entropy=lambda_entropy,
    max_steps=max_steps,
    rule_file_path=rule_filename,
    freeze_bert=freeze_bert,
    use_deq=use_deq,
    output_option=output_option
)

# Train

In [9]:
# train_model(model, train_loader, val_loader, epochs, lr, lambda_distill, save_path, use_amp=use_amp, lambda_energy=lambda_energy)

In [10]:
train_baseline(model, train_loader, val_loader, epochs, lr, save_path, use_amp=use_amp)

Using device: cuda
Enforced learning rate: 0.0001


Evaluate: 100%|██████████| 15/15 [00:08<00:00,  1.70it/s]


Epoch 0: Loss=0.9242 | Val F1=0.6698
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/MATRES/baseline_BERT.pt (F1: 0.6698)


Evaluate: 100%|██████████| 15/15 [00:09<00:00,  1.64it/s]


Epoch 1: Loss=0.8065 | Val F1=0.6825
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/MATRES/baseline_BERT.pt (F1: 0.6825)


...

Epoch 37: Loss=0.1064 | Val F1=0.7411


Evaluate: 100%|██████████| 15/15 [00:09<00:00,  1.62it/s]


Epoch 38: Loss=0.1020 | Val F1=0.7398


Evaluate: 100%|██████████| 15/15 [00:09<00:00,  1.59it/s]

Epoch 39: Loss=0.0948 | Val F1=0.7391
Baseline Training finished in 1609.33s.


In [11]:
# train_baseline_reasoning(model, train_loader, val_loader, epochs, lr, lambda_distill, save_path, use_amp=use_amp, lambda_energy=lambda_energy)

# Evaluate

In [12]:
print("Load pretrained model:")
checkpoint   = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])

Load pretrained model:


<All keys matched successfully>

In [13]:
cm, f1, p, r = evaluate_model(
    test_loader=test_loader,
    model=model
)
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

Evaluate: 100%|██████████| 7/7 [00:06<00:00,  1.01it/s]

CM:
 [[180  69   0  16]
 [ 41 356   0  16]
 [ 11  17   0   3]
 [ 27  75   0   6]]
f1: 0.6204997583736833 p 0.5928538670918269 r 0.6634026927784578


In [11]:
import pandas as pd
# Chạy dự đoán trên tập test và lưu ra file CSV (tùy chọn)
df_predictions = predict_and_save(model, test_loader, id2label, output_file="predictions.csv")
# Hiển thị kết quả
print(df_predictions.head())

Generating Predictions: 100%|██████████| 216/216 [03:01<00:00,  1.19it/s]

Predictions saved to predictions.csv
   doc_id E1_id E2_id Grouth-Truth Prediction
0     101   E84   E98      OVERLAP    OVERLAP
1     101   E71   E93      OVERLAP    OVERLAP
2     101  E129   E45      OVERLAP    OVERLAP
3     101   E46   E45       BEFORE    OVERLAP
4     101   E47   E45       BEFORE    OVERLAP
